# SARS plotting workflow

This notebook loads CSV outputs from `run_for_sars_modern.py` and generates modern summary figures.

## Imports and repo discovery

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "data_sars").exists() and (path / "scripts" / "sars_run").exists():
            return path
    raise RuntimeError("Could not locate Net_Dyns_Project_EpiRank repo root")


repo_root = find_repo_root(Path.cwd().resolve())
scripts_dir = repo_root / "scripts"
sars_run_dir = scripts_dir / "sars_run"
for path in [scripts_dir, sars_run_dir]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import plot_sars_results_modern as plot_modern

## Load result tables

In [ ]:
results_dir = repo_root / "results" / "sars"
movement_df, baseline_df, sensitivity_df, top_scores_df = plot_modern.load_results(results_dir)

{
    "movement_rows": len(movement_df),
    "baseline_rows": len(baseline_df),
    "sensitivity_rows": len(sensitivity_df),
    "top_score_rows": len(top_scores_df),
}

## Preview the result tables

In [ ]:
movement_df

In [ ]:
baseline_df

In [ ]:
sensitivity_df.sort_values("spearman", ascending=False).head(10)

In [ ]:
top_scores_df.head(15)

## Generate figures

In [ ]:
figures_dir = results_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
plot_modern.configure_plot_style()

combined_df = pd.concat([
    movement_df.assign(group="EpiRank movement setting"),
    baseline_df.assign(group="Baseline"),
], ignore_index=True)

plot_modern.make_metric_barplot(
    combined_df,
    "spearman",
    "Spearman Correlation Across EpiRank Settings and Baselines",
    figures_dir / "correlation_spearman_modern.png",
)
plot_modern.make_metric_barplot(
    combined_df,
    "pearson",
    "Pearson Correlation Across EpiRank Settings and Baselines",
    figures_dir / "correlation_pearson_modern.png",
)
plot_modern.make_sensitivity_heatmap(
    sensitivity_df,
    "spearman",
    "Sensitivity Heatmap (Spearman)",
    figures_dir / "sensitivity_spearman_modern.png",
)
plot_modern.make_sensitivity_heatmap(
    sensitivity_df,
    "pearson",
    "Sensitivity Heatmap (Pearson)",
    figures_dir / "sensitivity_pearson_modern.png",
)
plot_modern.make_top_scores_plot(
    top_scores_df,
    figures_dir / "top_scores_modern.png",
)

sorted(p.name for p in figures_dir.glob("*.png"))